# Smart Lender - Loan Prediction using Machine Learning

This notebook follows the Smart Lender workflow: importing libraries, reading the dataset, visualizing data, handling categorical and missing values, scaling, splitting, building models, evaluating performance, and saving the best model for Flask integration.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.style.use('fivethirtyeight')

## Import and Read Dataset

In [ ]:
data = pd.read_csv('../Dataset/loan_prediction.csv')
data.head()

In [ ]:
data.shape, data.info(), data.isnull().sum()

## Univariate Analysis

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(121)
sns.histplot(data['ApplicantIncome'], color='r', kde=True)
plt.subplot(122)
sns.histplot(data['Credit_History'], kde=True)
plt.show()

In [ ]:
plt.figure(figsize=(18, 4))
plt.subplot(1, 4, 1)
sns.countplot(y=data['Gender'])
plt.subplot(1, 4, 2)
sns.countplot(y=data['Education'])
plt.subplot(1, 4, 3)
sns.countplot(y=data['Married'])
plt.subplot(1, 4, 4)
sns.countplot(y=data['Loan_Status'])
plt.show()

## Bivariate and Multivariate Analysis

In [ ]:
plt.figure(figsize=(20, 5))
plt.subplot(131)
sns.countplot(x=data['Married'], hue=data['Gender'])
plt.subplot(132)
sns.countplot(x=data['Self_Employed'], hue=data['Education'])
plt.subplot(133)
sns.countplot(x=data['Property_Area'], hue=data['Loan_Amount_Term'])
plt.show()

In [ ]:
sns.swarmplot(x=data['Gender'], y=data['ApplicantIncome'], hue=data['Loan_Status'])
plt.show()

## Handling Categorical and Missing Values

In [ ]:
data = data.drop(columns=['Loan_ID'], errors='ignore')
data['Dependents'] = data['Dependents'].replace('3+', 3)

numeric_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')
    data[column] = data[column].fillna(data[column].mean())

for column in categorical_columns:
    data[column] = data[column].fillna(data[column].mode()[0])

In [ ]:
data['Gender'] = data['Gender'].map({'Male': 0, 'Female': 1})
data['Married'] = data['Married'].map({'No': 0, 'Yes': 1})
data['Education'] = data['Education'].map({'Not Graduate': 0, 'Graduate': 1})
data['Self_Employed'] = data['Self_Employed'].map({'No': 0, 'Yes': 1})
data['Property_Area'] = data['Property_Area'].map({'Rural': 0, 'Semiurban': 1, 'Urban': 2})
data['Loan_Status'] = data['Loan_Status'].map({'N': 0, 'Y': 1})
data['Dependents'] = pd.to_numeric(data['Dependents'], errors='coerce').fillna(0)
data.head()

## Scaling and Splitting Dataset

In [ ]:
x = data.drop(columns=['Loan_Status'])
y = data['Loan_Status']

names = x.columns
sc = StandardScaler()
x_scaled = sc.fit_transform(x)
x_scaled = pd.DataFrame(x_scaled, columns=names)

X_train, X_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.33, random_state=42, stratify=y)

## Model Building

In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_pred = model.predict(X_test)
    print(name)
    print('Training Accuracy:', accuracy_score(y_train, y_train_pred))
    print('Testing Accuracy:', accuracy_score(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    return accuracy_score(y_test, y_pred), model

models = [
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier()),
]

if XGBClassifier is not None:
    models.append(('XGBoost', XGBClassifier(eval_metric='logloss', random_state=42)))
else:
    print('XGBoost is not installed. Install using: python -m pip install xgboost')
    models.append(('Gradient Boosting Fallback', GradientBoostingClassifier(random_state=42)))

best_score = -1
best_model = None
for name, model in models:
    score, fitted = evaluate(name, model)
    if score > best_score:
        best_score = score
        best_model = fitted
    print('-' * 60)

## Saving the Model

In [ ]:
pickle.dump(best_model, open('../Flask/rdf.pkl', 'wb'))
pickle.dump(sc, open('../Flask/scale1.pkl', 'wb'))
print('Saved model and scaler successfully')

## Optional SMOTE Installation

If you want to include the dataset balancing step from the project description, install imbalanced-learn first:

```bash
python -m pip install imbalanced-learn
```

Then use `from imblearn.over_sampling import SMOTE` and apply `SMOTE().fit_resample(x, y)` before scaling/splitting.